> 在线讲义：[第 18 章 · Hopper ②：wgmma、warp specialization 与 persistent kernel](https://wikieden.github.io/cuda-to-tensor-core/lessons/18-hopper-wgmma-warp-specialization.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 18 章 · Hopper ②：wgmma、warp specialization 与 persistent kernel

本章回答：Hopper 的 Tensor Core 指令 `wgmma` 和 Ampere 的 `mma.sync` 有什么不同？为什么要把 block 里的 warp 分成"搬运组"和"计算组"？环形缓冲区 + full / empty 两组 barrier 怎么配合？persistent kernel 解决什么问题？一个 H100 GEMM 怎么从 32 TFLOP/s 做到超过 cuBLAS？

配套讲义：仓库里 `cuda-programming/notes/lessons/18-hopper-wgmma-warp-specialization.md`（讲解、动画、练习都在讲义里）。对应原文：Pranjal「Outperforming cuBLAS on H100: a Worklog」（`materials/text/blog-pranjal-outperforming-cublas-h100.txt` 全文 411 行，Kernel 1–11 在第 98–312 行）；Colfax「Fast Matrix-Multiplication with WGMMA on NVIDIA Hopper GPUs」（`text/blog-colfax-hopper-wgmma.txt`）；PTX §9.7.17 wgmma（`text/ptx-isa-9.4-focus-mma-wgmma-tcgen05-tma.txt` 第 10053–12039 行）；PG §4.12.1.3 Producer-Consumer Pattern Through Warp Specialization（`text/cuda-programming-guide.txt` 第 19533–19710 行）、§4.10.7（第 18294–18583 行）；FlashAttention-3 论文 §3.1（`text/flashattention-3.txt` 第 208–298 行）；GPU MODE L45（`text/gpu-mode-lecture-45-outperforming-cublas-on-h100.txt`）；Hopper tuning guide §4.1.2（`text/hopper-tuning-guide.txt` 第 160–185 行）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 m64n64 累加器 | 64 × 64 ÷ 128 | `accum_regs_per_thread(64, 64)` | 每线程 32 个 |
| 第三步 生产者 / 消费者 | full / empty 两组 barrier | `run_ring(20, 5, 2, 1)` | 消费者按顺序拿到 20 块；生产者快时要等 empty |
| 第四步 128 × 256 一个消费者 | 32768 ÷ 128 | `accum_regs_per_thread(128, 256)` | 256 > 255，溢出 |
| 第四步 寄存器再分配 | 240 × 128 × 2 + 24 × 128 | `register_budget(24, 240)` | 64512 ≤ 65536 |
| 第六步 全景 | ÷ cuBLAS 716 | `PRANJAL` | 4% → 107% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `CUBLAS_H100`**（来自 `ch18_warp_specialization.py`）

In [ ]:
CUBLAS_H100 = 716

**常量 `PRANJAL`**（来自 `ch18_warp_specialization.py`）

In [ ]:
PRANJAL = [(1, "CUDA core warptiling", 32), (2, "TMA + wgmma", 317), (3, "大 tile、大指令", 423),
           (4, "warp specialization", 498), (5, "2 个消费者 + 寄存器再分配", 631),
           (6, "persistent + tile 分组", 660), (7, "更快的 barrier", 704), (8, "cluster multicast", 734),
           ("9–10", "微优化、异步 store", 758), (11, "Hilbert 曲线", 764)]

**函数 `accum_regs_per_thread`**（来自 `ch18_warp_specialization.py`）：m × n 个 FP32 累加器分给一个 warpgroup（128 个线程），每个线程几个 32 位寄存器。

In [ ]:
def accum_regs_per_thread(m: int, n: int, threads: int = 128) -> int:
    """m × n 个 FP32 累加器分给一个 warpgroup（128 个线程），每个线程几个 32 位寄存器。"""
    return m * n // threads

**函数 `register_budget`**（来自 `ch18_warp_specialization.py`）：setmaxnreg 之后整个 block 要多少寄存器：1 个生产者 warpgroup + consumers 个消费者 warpgroup。

In [ ]:
def register_budget(producer_regs: int, consumer_regs: int, consumers: int = 2, threads: int = 128) -> int:
    """setmaxnreg 之后整个 block 要多少寄存器：1 个生产者 warpgroup + consumers 个消费者 warpgroup。"""
    return producer_regs * threads + consumer_regs * threads * consumers

**函数 `run_ring`**（来自 `ch18_warp_specialization.py`）：环形缓冲区：生产者等 empty[i] 再装 slot i，消费者等 full[i] 再用 slot i，用完 arrive empty[i]。

In [ ]:
def run_ring(n_tiles: int, qsize: int, producer_speed: int, consumer_speed: int) -> tuple[list[int], int]:
    """环形缓冲区：生产者等 empty[i] 再装 slot i，消费者等 full[i] 再用 slot i，用完 arrive empty[i]。

    两边按各自的速度（每"拍"能做几步）交替推进。返回消费者拿到的 tile 顺序，和生产者因为 slot 满而等的次数。
    """
    slots = [None] * qsize
    full = [False] * qsize           # full[i]：slot i 装满了
    empty = [True] * qsize           # 初始化：生产者"以为所有 slot 都是空的"
    produced = consumed = waits = 0
    got = []
    while consumed < n_tiles:
        for _ in range(producer_speed):
            if produced == n_tiles:
                break
            i = produced % qsize
            if not empty[i]:
                waits += 1
                break                # empty[i].wait()：这一拍没法装
            empty[i] = False
            slots[i] = produced      # TMA 搬 A_k、B_k → slot i
            full[i] = True           # TMA 完成时 full[i] 自动记账
            produced += 1
        for _ in range(consumer_speed):
            i = consumed % qsize
            if not full[i]:
                break                # full[i].wait()
            got.append(slots[i])     # wgmma 用 slot i
            full[i] = False
            empty[i] = True          # empty[i].arrive()
            consumed += 1
    return got, waits

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 wgmma m64n64k16 的累加器：每个线程", accum_regs_per_thread(64, 64), "个 FP32 寄存器")

In [ ]:
print("\n第三步 生产者 / 消费者，QSIZE = 5，20 块")
got, waits = run_ring(20, 5, producer_speed=2, consumer_speed=1)
print("  消费者按顺序拿到每一块：", got == list(range(20)), "；生产者因 slot 满等了", waits, "拍")
got, waits = run_ring(20, 5, producer_speed=1, consumer_speed=2)
print("  生产者慢时：顺序仍对", got == list(range(20)), "，生产者从不等 empty（等了", waits, "拍）")

In [ ]:
print("\n第四步 128 × 256 的输出 tile 放进一个 warpgroup")
print("  每线程", accum_regs_per_thread(128, 256), "个寄存器 > 上限 255 → 溢出")
b = register_budget(24, 240)
print(f"  寄存器再分配 24 / 240 / 240：240 × 128 × 2 + 24 × 128 = {b} ≤ 65536：{b <= 65536}")

In [ ]:
print("\n第六步 Pranjal 全景（H100 SXM、BF16，相对 cuBLAS 716 TFLOP/s）")
for k, what, t in PRANJAL:
    print(f"  Kernel {k:>2} {what:<16} {t:4d} TFLOP/s  {t / CUBLAS_H100:5.0%}")

## 3. 代码里才看得到的

1. **`empty` 初始全是 True**：对应讲义说的"初始化时让生产者以为所有 slot 都是空的"，否则第一块谁也开始不了。
2. **"一拍"是编的时间单位**：`producer_speed=2` 表示生产者每拍能装两块；只用来看"谁快谁等"，不对应真实周期。
3. **Kernel 8 算出来是 102.5%**：734 / 716 = 1.025，讲义表写 102%（截断），测试按 102–103 之间核对。

## 4. 自己改一改

- 把 `run_ring` 的 QSIZE 从 5 改成 1、2，生产者快的时候看它等了多少拍。
- 用 `register_budget` 试试生产者留 40 个、消费者 232 个，看还放不放得下。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> wgmma：四个 warp 一起、一条指令一大块、B 直接从 shared 读、发了就走；
> 分工：搬运组只发 TMA，计算组只做 wgmma；
> 环形传送带两盏灯：full 亮了消费者拿，empty 亮了生产者装；
> 寄存器劫贫济富：搬运组只留 24 个，计算组拿 240 个；
> 常驻 132 个 block 循环干：store 时就搬下一块，tile 排好顺序吃 L2。